# Reveal Atlas — онлайн-сервер из Google Drive

Этот notebook запускает **настоящий сервер онлайн-режима** Reveal Atlas
(Node.js + WebSocket) на виртуальной машине Google Colab и открывает к нему
**публичный HTTPS-адрес**. Notebook лежит в вашем Google Drive, поэтому
запускается в любой момент: «Открыть с помощью → Google Colaboratory» → «Среду выполнения» → «Выполнить все».

**Что важно понимать про Google Drive:** сам Диск — это хранилище файлов,
а не хостинг: он не умеет выполнять код и держать постоянные соединения
(веб-хостинг в Drive отключили ещё в 2016 году). Поэтому сервер запускается
в Colab, а Диск используется для двух вещей: хранения самого notebook и
(по желанию) сохранения комнат между запусками.

**Ограничения:** адрес `*.trycloudflare.com` временный — он живёт, пока
работает сессия Colab и сама ячейка. Для постоянной игры удобнее поднять
сервер на своём ПК (`npm run online`) или на VPS.

In [ ]:
# --- 0. Проверка, что всё на месте ---
import sys
print("Python:", sys.version.split()[0])
print("Ноутбук запущен в Colab:", "google.colab" in sys.modules)

In [ ]:
# --- 1. Подготовка: Node.js и cloudflared -------------------------------
# Этой ячейке нужно 20-40 секунд. Всё ставится внутрь Colab, ничего не меняется в системе.
import os, shutil, subprocess, sys, urllib.request, zipfile

WORK = '/content/reveal-atlas-online'
os.makedirs(f'{WORK}/server', exist_ok=True)

def sh(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

# Node.js 22 (в Colab его нет, а системный из apt слишком старый)
NODE_DIR = '/content/node'
if not os.path.exists(f'{NODE_DIR}/bin/node'):
    sh('wget -q https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz -O /content/node.tar.xz')
    sh('mkdir -p /content/node && tar -xf /content/node.tar.xz -C /content/node --strip-components=1')
os.environ['PATH'] = f'{NODE_DIR}/bin:' + os.environ['PATH']
print('node:', subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip())

# cloudflared — бесплатный туннель без регистрации
if not os.path.exists('/content/cloudflared'):
    sh('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared')
    sh('chmod +x /content/cloudflared')
print('cloudflared готов')


In [ ]:
# --- 2. Файлы сервера (встроены в notebook, интернет не нужен) -----------
# Это тот самый код из репозитория: WebSocket-сервер, комнаты, HTTP API.
import base64, os

FILES = {
  "ws.mjs": "LyoqCiAqINCc0LjQvdC40LzQsNC70YzQvdGL0LkgV2ViU29ja2V0LdGB0LXRgNCy0LXRgCDQv9C+IFJGQyA2NDU1INCx0LXQtyDQstC90LXRiNC90LjRhSDQt9Cw0LLQuNGB0LjQvNC+0YHRgtC10LkuCiAqINCd0YPQttC10L0sINGH0YLQvtCx0YsgwqvQvtC90LvQsNC50L0t0YDQtdC20LjQvMK7IFJldmVhbCBBdGxhcyDRgNCw0LHQvtGC0LDQuyDQvdCwINC70Y7QsdC+0LwgTm9kZS5qcwogKiDQsdC10Lcg0YPRgdGC0LDQvdC+0LLQutC4INC/0LDQutC10YLQvtCyINC40Lcg0LjQvdGC0LXRgNC90LXRgtCwLgogKi8KaW1wb3J0IHsgY3JlYXRlSGFzaCB9IGZyb20gJ25vZGU6Y3J5cHRvJzsKaW1wb3J0IHsgRXZlbnRFbWl0dGVyIH0gZnJvbSAnbm9kZTpldmVudHMnOwoKZXhwb3J0IGNvbnN0IFdTX0dVSUQgPSAnMjU4RUFGQTUtRTkxNC00N0RBLTk1Q0EtQzVBQjBEQzg1QjExJzsKZXhwb3J0IGNvbnN0IE1BWF9QQVlMT0FEX0JZVEVTID0gOCAqIDEwMjQgKiAxMDI0OwoKY29uc3QgT1BfQ09OVCA9IDB4MDsKY29uc3QgT1BfVEVYVCA9IDB4MTsKY29uc3QgT1BfQklOQVJZID0gMHgyOwpjb25zdCBPUF9DTE9TRSA9IDB4ODsKY29uc3QgT1BfUElORyA9IDB4OTsKY29uc3QgT1BfUE9ORyA9IDB4YTsKCi8qKiDQntGC0LLQtdGCINC90LAg0LfQsNCz0L7Qu9C+0LLQvtC6IFNlYy1XZWJTb2NrZXQtS2V5INC60LvQuNC10L3RgtCwLiAqLwpleHBvcnQgZnVuY3Rpb24gY29tcHV0ZUFjY2VwdEtleShrZXkpIHsKICByZXR1cm4gY3JlYXRlSGFzaCgnc2hhMScpLnVwZGF0ZShTdHJpbmcoa2V5KSArIFdTX0dVSUQpLmRpZ2VzdCgnYmFzZTY0Jyk7Cn0KCi8qKiDQntC00L3QviDQv9C+0LTQutC70Y7Rh9C10L3QuNC1OiDRgNCw0LfQsdC+0YAg0LLRhdC+0LTRj9GJ0LjRhSDQutCw0LTRgNC+0LIg0Lgg0L7RgtC/0YDQsNCy0LrQsCDRgdC+0L7QsdGJ0LXQvdC40LkuICovCmV4cG9ydCBjbGFzcyBXc0Nvbm5lY3Rpb24gZXh0ZW5kcyBFdmVudEVtaXR0ZXIgewogIGNvbnN0cnVjdG9yKHNvY2tldCkgewogICAgc3VwZXIoKTsKICAgIHRoaXMuc29ja2V0ID0gc29ja2V0OwogICAgdGhpcy5idWZmZXIgPSBCdWZmZXIuYWxsb2MoMCk7CiAgICB0aGlzLmZyYWdtZW50cyA9IFtdOwogICAgdGhpcy5mcmFnbWVudE9wY29kZSA9IDA7CiAgICB0aGlzLmNsb3NlZCA9IGZhbHNlOwogICAgdGhpcy5jbG9zZWRCeVBlZXIgPSBmYWxzZTsKICAgIHRoaXMuYWxpdmUgPSB0cnVlOwogICAgdGhpcy5yZW1vdGVBZGRyZXNzID0gc29ja2V0LnJlbW90ZUFkZHJlc3MgPz8gJyc7CiAgICAvKiog0J/RgNC+0LjQt9Cy0L7Qu9GM0L3Ri9C1INC00LDQvdC90YvQtSDQv9GA0LjQu9C+0LbQtdC90LjRjyAo0L3QsNC/0YDQuNC80LXRgCwg0LjQs9GA0L7QuikuICovCiAgICB0aGlzLmRhdGEgPSBudWxsOwoKICAgIHNvY2tldC5zZXROb0RlbGF5KHRydWUpOwogICAgdHJ5IHsKICAgICAgc29ja2V0LnNldEtlZXBBbGl2ZSh0cnVlLCAzMDAwMCk7CiAgICB9IGNhdGNoIHsKICAgICAgLyog0L3QtSDQstGB0LUg0YHQvtC60LXRgtGLINC/0L7QtNC00LXRgNC20LjQstCw0Y7RgiBrZWVwLWFsaXZlICovCiAgICB9CgogICAgc29ja2V0Lm9uKCdkYXRhJywgKGNodW5rKSA9PiB0aGlzLl9vbkRhdGEoY2h1bmspKTsKICAgIHNvY2tldC5vbignZXJyb3InLCAoKSA9PiB0aGlzLmRlc3Ryb3koKSk7CiAgICBzb2NrZXQub24oJ2Nsb3NlJywgKCkgPT4gdGhpcy5fb25Tb2NrZXRDbG9zZSgpKTsKICAgIHNvY2tldC5vbignZW5kJywgKCkgPT4gdGhpcy5fb25Tb2NrZXRDbG9zZSgpKTsKICB9CgogIF9vblNvY2tldENsb3NlKCkgewogICAgaWYgKHRoaXMuY2xvc2VkKSByZXR1cm47CiAgICB0aGlzLmNsb3NlZCA9IHRydWU7CiAgICB0aGlzLmVtaXQoJ2Nsb3NlJyk7CiAgfQoKICBfb25EYXRhKGNodW5rKSB7CiAgICBpZiAodGhpcy5jbG9zZWQpIHJldHVybjsKICAgIHRoaXMuYnVmZmVyID0gdGhpcy5idWZmZXIubGVuZ3RoID09PSAwID8gY2h1bmsgOiBCdWZmZXIuY29uY2F0KFt0aGlzLmJ1ZmZlciwgY2h1bmtdKTsKICAgIHRyeSB7CiAgICAgIHRoaXMuX3BhcnNlKCk7CiAgICB9IGNhdGNoIChlcnIpIHsKICAgICAgdGhpcy5jbG9zZSgxMDAyLCAncHJvdG9jb2wgZXJyb3InKTsKICAgICAgdGhpcy5lbWl0KCdlcnJvcicsIGVycik7CiAgICB9CiAgfQoKICBfcGFyc2UoKSB7CiAgICB3aGlsZSAoIXRoaXMuY2xvc2VkKSB7CiAgICAgIGNvbnN0IGJ1ZiA9IHRoaXMuYnVmZmVyOwogICAgICBpZiAoYnVmLmxlbmd0aCA8IDIpIHJldHVybjsKICAgICAgY29uc3QgYjAgPSBidWZbMF07CiAgICAgIGNvbnN0IGIxID0gYnVmWzFdOwogICAgICBjb25zdCBmaW4gPSAoYjAgJiAweDgwKSAhPT0gMDsKICAgICAgY29uc3Qgb3Bjb2RlID0gYjAgJiAweDBmOwogICAgICBjb25zdCBtYXNrZWQgPSAoYjEgJiAweDgwKSAhPT0gMDsKICAgICAgbGV0IGxlbmd0aCA9IGIxICYgMHg3ZjsKICAgICAgbGV0IG9mZnNldCA9IDI7CgogICAgICBpZiAobGVuZ3RoID09PSAxMjYpIHsKICAgICAgICBpZiAoYnVmLmxlbmd0aCA8IG9mZnNldCArIDIpIHJldHVybjsKICAgICAgICBsZW5ndGggPSBidWYucmVhZFVJbnQxNkJFKG9mZnNldCk7CiAgICAgICAgb2Zmc2V0ICs9IDI7CiAgICAgIH0gZWxzZSBpZiAobGVuZ3RoID09PSAxMjcpIHsKICAgICAgICBpZiAoYnVmLmxlbmd0aCA8IG9mZnNldCArIDgpIHJldHVybjsKICAgICAgICBjb25zdCBiaWcgPSBidWYucmVhZEJpZ1VJbnQ2NEJFKG9mZnNldCk7CiAgICAgICAgaWYgKGJpZyA+IEJpZ0ludChNQVhfUEFZTE9BRF9CWVRFUykpIHsKICAgICAgICAgIHRoaXMuY2xvc2UoMTAwOSwgJ21lc3NhZ2UgdG9vIGJpZycpOwogICAgICAgICAgcmV0dXJuOwogICAgICAgIH0KICAgICAgICBsZW5ndGggPSBOdW1iZXIoYmlnKTsKICAgICAgICBvZmZzZXQgKz0gODsKICAgICAgfQoKICAgICAgY29uc3QgbWFza09mZnNldCA9IG9mZnNldDsKICAgICAgaWYgKG1hc2tlZCkgewogICAgICAgIGlmIChidWYubGVuZ3RoIDwgb2Zmc2V0ICsgNCkgcmV0dXJuOwogICAgICAgIG9mZnNldCArPSA0OwogICAgICB9CiAgICAgIGlmIChsZW5ndGggPiBNQVhfUEFZTE9BRF9CWVRFUyB8fCBidWYubGVuZ3RoIDwgb2Zmc2V0ICsgbGVuZ3RoKSByZXR1cm47CgogICAgICBsZXQgcGF5bG9hZCA9IGJ1Zi5zdWJhcnJheShvZmZzZXQsIG9mZnNldCArIGxlbmd0aCk7CiAgICAgIGlmIChtYXNrZWQpIHsKICAgICAgICBjb25zdCBtYXNrID0gYnVmLnN1YmFycmF5KG1hc2tPZmZzZXQsIG1hc2tPZmZzZXQgKyA0KTsKICAgICAgICBjb25zdCBwbGFpbiA9IEJ1ZmZlci5hbGxvY1Vuc2FmZShsZW5ndGgpOwogICAgICAgIGZvciAobGV0IGkgPSAwOyBpIDwgbGVuZ3RoOyBpKyspIHBsYWluW2ldID0gcGF5bG9hZFtpXSBeIG1hc2tbaSAmIDNdOwogICAgICAgIHBheWxvYWQgPSBwbGFpbjsKICAgICAgfQoKICAgICAgdGhpcy5idWZmZXIgPSBidWYuc3ViYXJyYXkob2Zmc2V0ICsgbGVuZ3RoKTsKICAgICAgdGhpcy5faGFuZGxlRnJhbWUoZmluLCBvcGNvZGUsIHBheWxvYWQpOwogICAgfQogIH0KCiAgX2hhbmRsZUZyYW1lKGZpbiwgb3Bjb2RlLCBwYXlsb2FkKSB7CiAgICBpZiAob3Bjb2RlID09PSBPUF9QSU5HKSB7CiAgICAgIHRoaXMuX3NlbmRGcmFtZShPUF9QT05HLCBwYXlsb2FkKTsKICAgICAgcmV0dXJuOwogICAgfQogICAgaWYgKG9wY29kZSA9PT0gT1BfUE9ORykgewogICAgICB0aGlzLmFsaXZlID0gdHJ1ZTsKICAgICAgdGhpcy5lbWl0KCdwb25nJywgcGF5bG9hZCk7CiAgICAgIHJldHVybjsKICAgIH0KICAgIGlmIChvcGNvZGUgPT09IE9QX0NMT1NFKSB7CiAgICAgIHRoaXMuY2xvc2VkQnlQZWVyID0gdHJ1ZTsKICAgICAgbGV0IGNvZGUgPSAxMDA1OwogICAgICBsZXQgcmVhc29uID0gJyc7CiAgICAgIGlmIChwYXlsb2FkLmxlbmd0aCA+PSAyKSB7CiAgICAgICAgY29kZSA9IHBheWxvYWQucmVhZFVJbnQxNkJFKDApOwogICAgICAgIHJlYXNvbiA9IHBheWxvYWQuc3ViYXJyYXkoMikudG9TdHJpbmcoJ3V0ZjgnKTsKICAgICAgfQogICAgICB0cnkgewogICAgICAgIHRoaXMuX3NlbmRGcmFtZShPUF9DTE9TRSwgcGF5bG9hZC5zdWJhcnJheSgwLCBNYXRoLm1pbigyLCBwYXlsb2FkLmxlbmd0aCkpKTsKICAgICAgfSBjYXRjaCB7CiAgICAgICAgLyog0YHQvtC60LXRgiDRg9C20LUg0LzRkdGA0YLQsiAqLwogICAgICB9CiAgICAgIHRoaXMuY2xvc2VkID0gdHJ1ZTsKICAgICAgdGhpcy5lbWl0KCdjbG9zZWQnLCBjb2RlLCByZWFzb24pOwogICAgICB0aGlzLmVtaXQoJ2Nsb3NlJyk7CiAgICAgIHRoaXMuc29ja2V0LmVuZCgpOwogICAgICByZXR1cm47CiAgICB9CgogICAgLy8g0KPQv9GA0LDQstC70Y/RjtGJ0LjQtSDQutCw0LTRgNGLINC90LUg0LzQvtCz0YPRgiDQsdGL0YLRjCDRhNGA0LDQs9C80LXQvdGC0LjRgNC+0LLQsNC90Ysg0Lgg0LTQu9C40L3QvdC10LUgMTI1INCx0LDQudGCLgogICAgaWYgKG9wY29kZSA+PSAweDgpIHsKICAgICAgdGhpcy5jbG9zZSgxMDAyLCAnYmFkIGNvbnRyb2wgZnJhbWUnKTsKICAgICAgcmV0dXJuOwogICAgfQoKICAgIGlmIChvcGNvZGUgPT09IE9QX0NPTlQpIHsKICAgICAgaWYgKHRoaXMuZnJhZ21lbnRzLmxlbmd0aCA9PT0gMCkgewogICAgICAgIHRoaXMuY2xvc2UoMTAwMiwgJ3VuZXhwZWN0ZWQgY29udGludWF0aW9uJyk7CiAgICAgICAgcmV0dXJuOwogICAgICB9CiAgICB9IGVsc2UgewogICAgICBpZiAodGhpcy5mcmFnbWVudHMubGVuZ3RoID4gMCkgewogICAgICAgIHRoaXMuY2xvc2UoMTAwMiwgJ2V4cGVjdGVkIGNvbnRpbnVhdGlvbicpOwogICAgICAgIHJldHVybjsKICAgICAgfQogICAgICB0aGlzLmZyYWdtZW50T3Bjb2RlID0gb3Bjb2RlOwogICAgfQoKICAgIHRoaXMuZnJhZ21lbnRzLnB1c2gocGF5bG9hZCk7CiAgICBpZiAoIWZpbikgcmV0dXJuOwoKICAgIGNvbnN0IGZ1bGwgPSB0aGlzLmZyYWdtZW50cy5sZW5ndGggPT09IDEgPyB0aGlzLmZyYWdtZW50c1swXSA6IEJ1ZmZlci5jb25jYXQodGhpcy5mcmFnbWVudHMpOwogICAgY29uc3QgdHlwZSA9IHRoaXMuZnJhZ21lbnRPcGNvZGU7CiAgICB0aGlzLmZyYWdtZW50cyA9IFtdOwogICAgdGhpcy5mcmFnbWVudE9wY29kZSA9IDA7CiAgICB0aGlzLmFsaXZlID0gdHJ1ZTsKICAgIGlmICh0eXBlID09PSBPUF9URVhUKSB0aGlzLmVtaXQoJ21lc3NhZ2UnLCBmdWxsLnRvU3RyaW5nKCd1dGY4JyksICd0ZXh0Jyk7CiAgICBlbHNlIGlmICh0eXBlID09PSBPUF9CSU5BUlkpIHRoaXMuZW1pdCgnbWVzc2FnZScsIGZ1bGwsICdiaW5hcnknKTsKICAgIGVsc2UgdGhpcy5jbG9zZSgxMDAyLCAnYmFkIG9wY29kZScpOwogIH0KCiAgX3NlbmRGcmFtZShvcGNvZGUsIHBheWxvYWQpIHsKICAgIGlmICh0aGlzLmNsb3NlZCB8fCB0aGlzLnNvY2tldC5kZXN0cm95ZWQpIHJldHVybiBmYWxzZTsKICAgIGNvbnN0IGxlbmd0aCA9IHBheWxvYWQubGVuZ3RoOwogICAgbGV0IGhlYWRlclNpemU7CiAgICBpZiAobGVuZ3RoIDwgMTI2KSBoZWFkZXJTaXplID0gMjsKICAgIGVsc2UgaWYgKGxlbmd0aCA8IDY1NTM2KSBoZWFkZXJTaXplID0gNDsKICAgIGVsc2UgaGVhZGVyU2l6ZSA9IDEwOwogICAgY29uc3QgZnJhbWUgPSBCdWZmZXIuYWxsb2NVbnNhZmUoaGVhZGVyU2l6ZSArIGxlbmd0aCk7CiAgICBmcmFtZVswXSA9IDB4ODAgfCBvcGNvZGU7CiAgICBpZiAobGVuZ3RoIDwgMTI2KSB7CiAgICAgIGZyYW1lWzFdID0gbGVuZ3RoOwogICAgfSBlbHNlIGlmIChsZW5ndGggPCA2NTUzNikgewogICAgICBmcmFtZVsxXSA9IDEyNjsKICAgICAgZnJhbWUud3JpdGVVSW50MTZCRShsZW5ndGgsIDIpOwogICAgfSBlbHNlIHsKICAgICAgZnJhbWVbMV0gPSAxMjc7CiAgICAgIGZyYW1lLndyaXRlQmlnVUludDY0QkUoQmlnSW50KGxlbmd0aCksIDIpOwogICAgfQogICAgcGF5bG9hZC5jb3B5KGZyYW1lLCBoZWFkZXJTaXplKTsKICAgIHRyeSB7CiAgICAgIHJldHVybiB0aGlzLnNvY2tldC53cml0ZShmcmFtZSk7CiAgICB9IGNhdGNoIHsKICAgICAgdGhpcy5kZXN0cm95KCk7CiAgICAgIHJldHVybiBmYWxzZTsKICAgIH0KICB9CgogIC8qKiDQntGC0L/RgNCw0LLQu9GP0LXRgiDRgtC10LrRgdGCICjQvtCx0YrQtdC60YLRiyDRgdC10YDQuNCw0LvQuNC30YPRjtGC0YHRjyDQsiBKU09OKS4gKi8KICBzZW5kKGRhdGEpIHsKICAgIGlmICh0eXBlb2YgZGF0YSA9PT0gJ3N0cmluZycpIHJldHVybiB0aGlzLl9zZW5kRnJhbWUoT1BfVEVYVCwgQnVmZmVyLmZyb20oZGF0YSwgJ3V0ZjgnKSk7CiAgICByZXR1cm4gdGhpcy5fc2VuZEZyYW1lKE9QX1RFWFQsIEJ1ZmZlci5mcm9tKEpTT04uc3RyaW5naWZ5KGRhdGEpLCAndXRmOCcpKTsKICB9CgogIHBpbmcocGF5bG9hZCA9IEJ1ZmZlci5hbGxvYygwKSkgewogICAgcmV0dXJuIHRoaXMuX3NlbmRGcmFtZShPUF9QSU5HLCBwYXlsb2FkKTsKICB9CgogIGNsb3NlKGNvZGUgPSAxMDAwLCByZWFzb24gPSAnJykgewogICAgaWYgKHRoaXMuY2xvc2VkKSByZXR1cm47CiAgICBjb25zdCByZWFzb25CdWYgPSBCdWZmZXIuZnJvbShTdHJpbmcocmVhc29uKS5zbGljZSgwLCAxMjApLCAndXRmOCcpOwogICAgY29uc3QgcGF5bG9hZCA9IEJ1ZmZlci5hbGxvY1Vuc2FmZSgyICsgcmVhc29uQnVmLmxlbmd0aCk7CiAgICBwYXlsb2FkLndyaXRlVUludDE2QkUoY29kZSwgMCk7CiAgICByZWFzb25CdWYuY29weShwYXlsb2FkLCAyKTsKICAgIHRyeSB7CiAgICAgIHRoaXMuX3NlbmRGcmFtZShPUF9DTE9TRSwgcGF5bG9hZCk7CiAgICB9IGNhdGNoIHsKICAgICAgLyogaWdub3JlICovCiAgICB9CiAgICB0aGlzLmNsb3NlZCA9IHRydWU7CiAgICB0aGlzLnNvY2tldC5lbmQoKTsKICAgIHNldFRpbWVvdXQoKCkgPT4gdGhpcy5kZXN0cm95KCksIDEwMDApLnVucmVmKCk7CiAgfQoKICBkZXN0cm95KCkgewogICAgaWYgKHRoaXMuY2xvc2VkKSB7CiAgICAgIHRyeSB7CiAgICAgICAgdGhpcy5zb2NrZXQuZGVzdHJveSgpOwogICAgICB9IGNhdGNoIHsKICAgICAgICAvKiBpZ25vcmUgKi8KICAgICAgfQogICAgICByZXR1cm47CiAgICB9CiAgICB0aGlzLmNsb3NlZCA9IHRydWU7CiAgICB0cnkgewogICAgICB0aGlzLnNvY2tldC5kZXN0cm95KCk7CiAgICB9IGNhdGNoIHsKICAgICAgLyogaWdub3JlICovCiAgICB9CiAgICB0aGlzLmVtaXQoJ2Nsb3NlJyk7CiAgfQp9CgovKiog0J/RgNC40L3QuNC80LDQtdGCIFdlYlNvY2tldC3Qv9C+0LTQutC70Y7Rh9C10L3QuNGPINC90LAg0YPQttC1INGB0YPRidC10YHRgtCy0YPRjtGJ0LXQvCBodHRwLlNlcnZlci4gKi8KZXhwb3J0IGNsYXNzIFdzU2VydmVyIGV4dGVuZHMgRXZlbnRFbWl0dGVyIHsKICBjb25zdHJ1Y3RvcihvcHRpb25zID0ge30pIHsKICAgIHN1cGVyKCk7CiAgICB0aGlzLnBhdGggPSBvcHRpb25zLnBhdGggPz8gbnVsbDsKICAgIHRoaXMuY29ubmVjdGlvbnMgPSBuZXcgU2V0KCk7CiAgICB0aGlzLl9oZWFydGJlYXQgPSBudWxsOwogICAgdGhpcy5faW50ZXJ2YWwgPSBvcHRpb25zLmhlYXJ0YmVhdE1zID8/IDI1MDAwOwogIH0KCiAgLyoqINCS0LXRiNCw0LXRgtGB0Y8g0L3QsCDRgdC+0LHRi9GC0LjQtSAndXBncmFkZScgaHR0cC5TZXJ2ZXIuICovCiAgaGFuZGxlVXBncmFkZShyZXEsIHNvY2tldCwgaGVhZCkgewogICAgY29uc3QgdXBncmFkZSA9IFN0cmluZyhyZXEuaGVhZGVycy51cGdyYWRlID8/ICcnKS50b0xvd2VyQ2FzZSgpOwogICAgY29uc3Qga2V5ID0gcmVxLmhlYWRlcnNbJ3NlYy13ZWJzb2NrZXQta2V5J107CiAgICBpZiAodXBncmFkZSAhPT0gJ3dlYnNvY2tldCcgfHwgIWtleSB8fCBTdHJpbmcocmVxLmhlYWRlcnNbJ3NlYy13ZWJzb2NrZXQtdmVyc2lvbiddID8/ICcnKSAhPT0gJzEzJykgewogICAgICBzb2NrZXQuZGVzdHJveSgpOwogICAgICByZXR1cm47CiAgICB9CiAgICBpZiAodGhpcy5wYXRoKSB7CiAgICAgIGNvbnN0IHBhdGggPSBTdHJpbmcocmVxLnVybCA/PyAnLycpLnNwbGl0KCc/JylbMF07CiAgICAgIGlmIChwYXRoICE9PSB0aGlzLnBhdGgpIHsKICAgICAgICBzb2NrZXQuZGVzdHJveSgpOwogICAgICAgIHJldHVybjsKICAgICAgfQogICAgfQoKICAgIHNvY2tldC53cml0ZSgKICAgICAgYEhUVFAvMS4xIDEwMSBTd2l0Y2hpbmcgUHJvdG9jb2xzXHJcbmAgKwogICAgICAgIGBVcGdyYWRlOiB3ZWJzb2NrZXRcclxuYCArCiAgICAgICAgYENvbm5lY3Rpb246IFVwZ3JhZGVcclxuYCArCiAgICAgICAgYFNlYy1XZWJTb2NrZXQtQWNjZXB0OiAke2NvbXB1dGVBY2NlcHRLZXkoa2V5KX1cclxuXHJcbmAsCiAgICApOwoKICAgIGNvbnN0IGNvbm4gPSBuZXcgV3NDb25uZWN0aW9uKHNvY2tldCk7CiAgICB0aGlzLmNvbm5lY3Rpb25zLmFkZChjb25uKTsKICAgIGNvbm4ub24oJ2Nsb3NlJywgKCkgPT4gdGhpcy5jb25uZWN0aW9ucy5kZWxldGUoY29ubikpOwogICAgaWYgKGhlYWQgJiYgaGVhZC5sZW5ndGggPiAwKSBjb25uLl9vbkRhdGEoaGVhZCk7CiAgICB0aGlzLmVtaXQoJ2Nvbm5lY3Rpb24nLCBjb25uLCByZXEpOwogIH0KCiAgLyoqINCf0LjQvdCz0YPQtdGCINCy0YHQtdGFINC4INC+0YLQutC70Y7Rh9Cw0LXRgiDQvNC+0LvRh9Cw0YnQuNC1INC/0L7QtNC60LvRjtGH0LXQvdC40Y8uICovCiAgc3RhcnRIZWFydGJlYXQoaW50ZXJ2YWxNcyA9IHRoaXMuX2ludGVydmFsLCB0aW1lb3V0TXMgPSBpbnRlcnZhbE1zICogMi41KSB7CiAgICB0aGlzLnN0b3BIZWFydGJlYXQoKTsKICAgIHRoaXMuX2hlYXJ0YmVhdCA9IHNldEludGVydmFsKCgpID0+IHsKICAgICAgZm9yIChjb25zdCBjb25uIG9mIFsuLi50aGlzLmNvbm5lY3Rpb25zXSkgewogICAgICAgIGlmICghY29ubi5hbGl2ZSkgewogICAgICAgICAgY29ubi5jbG9zZSgxMDAxLCAndGltZW91dCcpOwogICAgICAgICAgY29udGludWU7CiAgICAgICAgfQogICAgICAgIGNvbm4uYWxpdmUgPSBmYWxzZTsKICAgICAgICBjb25uLnBpbmcoKTsKICAgICAgfQogICAgICB2b2lkIHRpbWVvdXRNczsKICAgIH0sIGludGVydmFsTXMpOwogICAgaWYgKHRoaXMuX2hlYXJ0YmVhdC51bnJlZikgdGhpcy5faGVhcnRiZWF0LnVucmVmKCk7CiAgfQoKICBzdG9wSGVhcnRiZWF0KCkgewogICAgaWYgKHRoaXMuX2hlYXJ0YmVhdCkgY2xlYXJJbnRlcnZhbCh0aGlzLl9oZWFydGJlYXQpOwogICAgdGhpcy5faGVhcnRiZWF0ID0gbnVsbDsKICB9CgogIGJyb2FkY2FzdChkYXRhLCBleGNlcHQpIHsKICAgIGNvbnN0IHRleHQgPSB0eXBlb2YgZGF0YSA9PT0gJ3N0cmluZycgPyBkYXRhIDogSlNPTi5zdHJpbmdpZnkoZGF0YSk7CiAgICBmb3IgKGNvbnN0IGNvbm4gb2YgWy4uLnRoaXMuY29ubmVjdGlvbnNdKSB7CiAgICAgIGlmIChleGNlcHQgJiYgY29ubiA9PT0gZXhjZXB0KSBjb250aW51ZTsKICAgICAgY29ubi5zZW5kKHRleHQpOwogICAgfQogIH0KCiAgY2xvc2VBbGwoY29kZSA9IDEwMDEsIHJlYXNvbiA9ICdzZXJ2ZXIgc2h1dGRvd24nKSB7CiAgICBmb3IgKGNvbnN0IGNvbm4gb2YgWy4uLnRoaXMuY29ubmVjdGlvbnNdKSBjb25uLmNsb3NlKGNvZGUsIHJlYXNvbik7CiAgfQp9Cg==",
  "rooms.mjs": "...",
  "online.mjs": "...",
  "index.mjs": "IyEvdXNyL2Jpbi9lbnYgbm9kZQovKioKICog0JDQstGC0L7QvdC+0LzQvdGL0Lkg0YHQtdGA0LLQtdGAIFJldmVhbCBBdGxhczog0YDQsNC30LTQsNGR0YIg0YHQvtCx0YDQsNC90L3QvtC1INC/0YDQuNC70L7QttC10L3QuNC1INC40LcgZGlzdC8KICog0Lgg0L/QvtC00L3QuNC80LDQtdGCINC90LDRgdGC0L7Rj9GJ0LjQuSDQvtC90LvQsNC50L0t0YDQtdC20LjQvCAoV2ViU29ja2V0LCDQutC+0LzQvdCw0YLRiywg0L7QsdGJ0LjQuSDQv9GA0L7Qs9GA0LXRgdGBKS4KICoKICogICBub2RlIHNlcnZlci9pbmRleC5tanMgICAgICAgICAgICAgICAgICAgICAgICAjINC/0L7RgNGCIDgwODAsINC00LDQvdC90YvQtSDQsiAuL2RhdGEvb25saW5lCiAqICAgbm9kZSBzZXJ2ZXIvaW5kZXgubWpzIC0tcG9ydCAzMDAwCiAqICAgbm9kZSBzZXJ2ZXIvaW5kZXgubWpzIC0taG9zdCAwLjAuMC4wIC0tZGF0YSAvc3J2L2F0bGFzIC0tZGlzdCAuL2Rpc3QKICovCmltcG9ydCBodHRwIGZyb20gJ25vZGU6aHR0cCc7CmltcG9ydCBmcyBmcm9tICdub2RlOmZzJzsKaW1wb3J0IHBhdGggZnJvbSAnbm9kZTpwYXRoJzsKaW1wb3J0IHsgZmlsZVVSTFRvUGF0aCB9IGZyb20gJ25vZGU6dXJsJzsKaW1wb3J0IHsgYXR0YWNoT25saW5lU2VydmVyIH0gZnJvbSAnLi9vbmxpbmUubWpzJzsKCmNvbnN0IF9fZGlybmFtZSA9IHBhdGguZGlybmFtZShmaWxlVVJMVG9QYXRoKGltcG9ydC5tZXRhLnVybCkpOwpjb25zdCBST09UID0gcGF0aC5yZXNvbHZlKF9fZGlybmFtZSwgJy4uJyk7CgpmdW5jdGlvbiBwYXJzZUFyZ3MoYXJndikgewogIGNvbnN0IG9wdGlvbnMgPSB7CiAgICBwb3J0OiBOdW1iZXIocHJvY2Vzcy5lbnYuUE9SVCkgfHwgODA4MCwKICAgIGhvc3Q6IHByb2Nlc3MuZW52LkhPU1QgfHwgJzAuMC4wLjAnLAogICAgZGF0YTogbnVsbCwKICAgIGRpc3Q6IG51bGwsCiAgfTsKICBmb3IgKGxldCBpID0gMjsgaSA8IGFyZ3YubGVuZ3RoOyBpKyspIHsKICAgIGNvbnN0IGFyZyA9IGFyZ3ZbaV07CiAgICBpZiAoYXJnID09PSAnLS1wb3J0JyB8fCBhcmcgPT09ICctcCcpIG9wdGlvbnMucG9ydCA9IE51bWJlcihhcmd2WysraV0pOwogICAgZWxzZSBpZiAoYXJnID09PSAnLS1ob3N0Jykgb3B0aW9ucy5ob3N0ID0gYXJndlsrK2ldOwogICAgZWxzZSBpZiAoYXJnID09PSAnLS1kYXRhJyB8fCBhcmcgPT09ICctZCcpIG9wdGlvbnMuZGF0YSA9IGFyZ3ZbKytpXTsKICAgIGVsc2UgaWYgKGFyZyA9PT0gJy0tZGlzdCcpIG9wdGlvbnMuZGlzdCA9IGFyZ3ZbKytpXTsKICAgIGVsc2UgaWYgKGFyZyA9PT0gJy0taGVscCcpIHsKICAgICAgY29uc29sZS5sb2coCiAgICAgICAgJ9CY0YHQv9C+0LvRjNC30L7QstCw0L3QuNC1OiBub2RlIHNlcnZlci9pbmRleC5tanMgWy0tcG9ydCA4MDgwXSBbLS1ob3N0IDAuMC4wLjBdIFstLWRhdGEgLi9kYXRhL29ubGluZV0gWy0tZGlzdCAuL2Rpc3RdJywKICAgICAgKTsKICAgICAgcHJvY2Vzcy5leGl0KDApOwogICAgfQogIH0KICByZXR1cm4gb3B0aW9uczsKfQoKY29uc3QgYXJncyA9IHBhcnNlQXJncyhwcm9jZXNzLmFyZ3YpOwpjb25zdCBkaXN0RGlyID0gcGF0aC5yZXNvbHZlKGFyZ3MuZGlzdCA/PyBwYXRoLmpvaW4oUk9PVCwgJ2Rpc3QnKSk7CmNvbnN0IGRhdGFEaXIgPSBwYXRoLnJlc29sdmUoYXJncy5kYXRhID8/IHBhdGguam9pbihST09ULCAnZGF0YScsICdvbmxpbmUnKSk7CgppZiAoIWZzLmV4aXN0c1N5bmMocGF0aC5qb2luKGRpc3REaXIsICdpbmRleC5odG1sJykpKSB7CiAgY29uc29sZS5lcnJvcihg0J3QtSDQvdCw0LnQtNC10L3QviDQv9GA0LjQu9C+0LbQtdC90LjQtSDQsiAke2Rpc3REaXJ9LmApOwogIGNvbnNvbGUuZXJyb3IoJ9Ch0L3QsNGH0LDQu9CwINGB0L7QsdC10YDQuNGC0LUg0L/RgNC+0LXQutGCOiBucG0gcnVuIGJ1aWxkJyk7CiAgcHJvY2Vzcy5leGl0KDEpOwp9Cgpmcy5ta2RpclN5bmMoZGF0YURpciwgeyByZWN1cnNpdmU6IHRydWUgfSk7Cgpjb25zdCBodHRwU2VydmVyID0gaHR0cC5jcmVhdGVTZXJ2ZXIoKTsKaHR0cFNlcnZlci5vbignY2xpZW50RXJyb3InLCAoX2Vyciwgc29ja2V0KSA9PiB7CiAgaWYgKHNvY2tldC53cml0YWJsZSkgc29ja2V0LmVuZCgnSFRUUC8xLjEgNDAwIEJhZCBSZXF1ZXN0XHJcblxyXG4nKTsKfSk7Cgpjb25zdCBvbmxpbmUgPSBhdHRhY2hPbmxpbmVTZXJ2ZXIoaHR0cFNlcnZlciwgeyBkYXRhRGlyLCBzdGF0aWNSb290OiBkaXN0RGlyIH0pOwoKaHR0cFNlcnZlci5saXN0ZW4oYXJncy5wb3J0LCBhcmdzLmhvc3QsICgpID0+IHsKICBjb25zdCBhZGRyZXNzID0gaHR0cFNlcnZlci5hZGRyZXNzKCk7CiAgY29uc3QgcG9ydCA9IHR5cGVvZiBhZGRyZXNzID09PSAnb2JqZWN0JyAmJiBhZGRyZXNzID8gYWRkcmVzcy5wb3J0IDogYXJncy5wb3J0OwogIGNvbnN0IHNob3duSG9zdCA9IGFyZ3MuaG9zdCA9PT0gJzAuMC4wLjAnID8gJ2xvY2FsaG9zdCcgOiBhcmdzLmhvc3Q7CiAgY29uc29sZS5sb2coJycpOwogIGNvbnNvbGUubG9nKCcgIFJldmVhbCBBdGxhcyDigJQg0L7QvdC70LDQudC9LdGB0LXRgNCy0LXRgCDQt9Cw0L/Rg9GJ0LXQvScpOwogIGNvbnNvbGUubG9nKGAgINCf0YDQuNC70L7QttC10L3QuNC1OiAgIGh0dHA6Ly8ke3Nob3duSG9zdH06JHtwb3J0fS9gKTsKICBjb25zb2xlLmxvZyhgICDQntC90LvQsNC50L0gKFdTKTogIHdzOi8vJHtzaG93bkhvc3R9OiR7cG9ydH0vd3NgKTsKICBjb25zb2xlLmxvZyhgICDQlNCw0L3QvdGL0LU6ICAgICAgICR7ZGF0YURpcn1gKTsKICBjb25zb2xlLmxvZygnJyk7CiAgY29uc29sZS5sb2coJyAg0J7RgtC60YDQvtC50YLQtSDQv9GA0LjQu9C+0LbQtdC90LjQtSDQsiDQtNCy0YPRhSDQvtC60L3QsNGFICjQuNC70Lgg0L7RgtC/0YDQsNCy0YzRgtC1INGB0YHRi9C70LrRgyDQtNGA0YPQs9GDKSwnKTsKICBjb25zb2xlLmxvZygnICDRgdC+0LfQtNCw0LnRgtC1INC60L7QvNC90LDRgtGDINCyINC+0LTQvdC+0Lwg0Lgg0L/QvtC00LrQu9GO0YfQuNGC0LXRgdGMINC/0L4g0LrQvtC00YMg0LIg0LTRgNGD0LPQvtC8LicpOwogIGNvbnNvbGUubG9nKCcnKTsKfSk7CgpmdW5jdGlvbiBzaHV0ZG93bihzaWduYWwpIHsKICBjb25zb2xlLmxvZyhgXG7Qn9C+0LvRg9GH0LXQvSAke3NpZ25hbH0sINGB0L7RhdGA0LDQvdGP0LXQvCDQutC+0LzQvdCw0YLRiyDQuCDQstGL0YXQvtC00LjQvOKApmApOwogIG9ubGluZS5zdG9wKCk7CiAgaHR0cFNlcnZlci5jbG9zZSgoKSA9PiBwcm9jZXNzLmV4aXQoMCkpOwogIHNldFRpbWVvdXQoKCkgPT4gcHJvY2Vzcy5leGl0KDApLCAxNTAwKS51bnJlZigpOwp9Cgpwcm9jZXNzLm9uKCdTSUdJTlQnLCAoKSA9PiBzaHV0ZG93bignU0lHSU5UJykpOwpwcm9jZXNzLm9uKCdTSUdURVJNJywgKCkgPT4gc2h1dGRvd24oJ1NJR1RFUk0nKSk7Cg=="
}

WORK = '/content/reveal-atlas-online'
for name, b64 in FILES.items():
    path = os.path.join(WORK, 'server', name)
    with open(path, 'wb') as f:
        f.write(base64.b64decode(b64))
    print('записан', path)


In [ ]:
# --- 3. Запуск: сервер + публичный адрес --------------------------------
# Оставьте эту ячейку работающей, пока идёт игра. Остановка (■) гасит сервер.
import os, re, shutil, subprocess, sys, threading, time
from IPython.display import HTML, display

PORT = 8080                 # порт внутри Colab (наружу откроется туннель)
USE_DRIVE = True            # True — комнаты сохраняются в вашем Google Drive
BUILD_APP = False           # True — собрать и раздавать само приложение (нужен git-доступ к репозиторию)
REPO_URL = ''               # например 'https://github.com/owlclockl/sic.git' — только для BUILD_APP

WORK = '/content/reveal-atlas-online'
os.environ['PATH'] = '/content/node/bin:' + os.environ['PATH']

# Куда складывать комнаты и картинки миров.
DATA_DIR = f'{WORK}/data'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA_DIR = '/content/drive/MyDrive/RevealAtlas-Online'
        os.makedirs(DATA_DIR, exist_ok=True)
        print('Комнаты будут храниться в Google Drive:', DATA_DIR)
    except Exception as err:
        print('Не удалось подключить Google Drive, храним внутри Colab:', err)
        USE_DRIVE = False

if BUILD_APP and REPO_URL:
    subprocess.run(f'rm -rf {WORK}/app && git clone --depth 1 {REPO_URL} {WORK}/app', shell=True, check=True)
    subprocess.run(f'cd {WORK}/app && npm ci && npm run build', shell=True, check=True)
    APP_DIST = f'{WORK}/app/dist'
else:
    APP_DIST = f'{WORK}/dist'
    if not os.path.exists(f'{APP_DIST}/index.html'):
        APP_DIST = ''   # приложение откроется с любого другого адреса, сервер всё равно работает

# Сервер
server = subprocess.Popen(
    ['node', 'server/index.mjs', '--port', str(PORT), '--data', DATA_DIR] + (['--dist', APP_DIST] if APP_DIST else []),
    cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

def pump_server():
    for line in iter(server.stdout.readline, ''):
        print('[сервер]', line, end='')

threading.Thread(target=pump_server, daemon=True).start()

# Публичный адрес
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if found:
        public_url = found.group(0)
        break

if not public_url:
    print('Не удалось получить публичный адрес. Сервер всё равно работает локально внутри Colab.')
else:
    print()
    print('=' * 72)
    print('  ПУБЛИЧНЫЙ АДРЕС СЕРВЕРА:', public_url)
    print('=' * 72)
    print()
    if APP_DIST:
        print('  1. Откройте приложение:', public_url + '/')
    else:
        print('  1. Откройте Reve Atlas на любом устройстве (свой ПК, телефон, ReveAtlas.html).')
    print('  2. Нажмите «Онлайн» → в поле «Сервер» вставьте:', public_url)
    print('  3. «+ Создать комнату» и отправьте другу код или ссылку-приглашение.')
    print()
    display(HTML(f'<a href="{public_url}/" target="_blank" style="font-size:16px">Открыть Reve Atlas: {public_url}</a>'))

# Держим сессию живой, пока оба процесса работают.
try:
    while server.poll() is None and tunnel.poll() is None:
        time.sleep(2)
finally:
    print('Останавливаем сервер и туннель…')
    for proc in (tunnel, server):
        if proc.poll() is None:
            proc.terminate()


## Как играть

1. Друзья открывают приложение Reve Atlas (свой ПК, телефон или файл `RevealAtlas.html`).
2. В панели «Онлайн» в поле «Сервер» вставляют публичный адрес из предыдущей ячейки.
3. Один создаёт комнату, остальные входят по коду или ссылке-приглашению.
4. Все мазки, курсоры, чат и смена сетки синхронны — прогресс общий.

## Если адрес не появился

- Cloudflare иногда отвечает не сразу: перезапустите последнюю ячейку.
- В меню Colab проверьте, что сессия не отключилась из-за простоя.
- Альтернатива без туннеля: запустите сервер на ПК (`npm run online`) и
  откройте `http://<IP-вашего-ПК>:8080` — в локальной сети это работает сразу.